# Icecream Sales & Temperature ML Pipeline

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

In [36]:
data = pd.read_csv("Ice Cream Sales and Temperature.csv")

In [37]:
data.head()

,Date,Time,Temperature (Celsius),Ice Cream Sales
0,2023-01-01,08:00,18,20
1,2023-01-01,08:05,19,22
2,2023-01-01,08:10,20,25
3,2023-01-01,08:15,21,24
4,2023-01-01,08:20,22,26


In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Date                   45 non-null     object
 1   Time                   45 non-null     object
 2   Temperature (Celsius)  45 non-null     int64 
 3   Ice Cream Sales        45 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.5+ KB


In [39]:
data.isnull().sum()

Date                     0
Time                     0
Temperature (Celsius)    0
Ice Cream Sales          0
dtype: int64

In [40]:
data.duplicated().sum()

np.int64(0)

In [41]:
data.drop_duplicates(inplace=True)
data.reset_index(drop=True, inplace=True)

In [42]:
data['Date'] = pd.to_datetime(data['Date'])
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['Time'] = pd.to_datetime(
    data['Time'],
    format='%H:%M'
)
data['Hour'] = data['Time'].dt.hour
data['Minute'] = data['Time'].dt.minute
data.drop(['Date', 'Time'], axis=1, inplace=True)

In [43]:
X = data.drop('Ice Cream Sales', axis=1)
y = data['Ice Cream Sales']

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [45]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [46]:
linear_pipeline = Pipeline([
    ('preprocessing', num_pipeline),
    ('model', LinearRegression())
])

In [54]:
linear_pipeline.fit(X_train, y_train)
linear_pred = linear_pipeline.predict(X_test)
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_r2 = r2_score(y_test, linear_pred)

In [50]:
poly_pipeline = Pipeline([
    ('preprocessing', num_pipeline),
    ('polynomial', PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ('model', LinearRegression())
])

In [53]:
poly_pipeline.fit(X_train, y_train)
poly_pred = poly_pipeline.predict(X_test)
poly_mae = mean_absolute_error(y_test, poly_pred)
poly_r2 = r2_score(y_test, poly_pred)

In [56]:
print(f"""
Linear r2 Score: {linear_r2}
Linear MAE: {linear_mae}

Polynomial r2 Score: {poly_r2}
Polynomial MAE: {poly_mae}
""")


Linear r2 Score: 0.8224318728761153
Linear MAE: 4.601137549371692

Polynomial r2 Score: 0.9203032057751656
Polynomial MAE: 2.361501792110803



In [57]:
final_model = poly_pipeline
def predict_sales(date, time, temperature):

    date = pd.to_datetime(date)
    time = pd.to_datetime(time, format='%H:%M')

    new_data = pd.DataFrame({
        'Temperature (Celsius)': [temperature],
        'Year': [date.year],
        'Month': [date.month],
        'Day': [date.day],
        'Hour': [time.hour],
        'Minute': [time.minute]
    })

    prediction = final_model.predict(new_data)

    return round(float(prediction[0]), 2)

In [58]:
predict_sales(
    date='2023-01-01',
    time='08:30',
    temperature=23
)

31.04